# TP 1 — Du texte brut au premier modèle bigramme

Objectif : comprendre comment on passe d'un corpus de texte à des données d'entraînement pour entraîner une petite table de prédictions (bigramme).

**À rendre** : le notebook complet exécuté de bout en bout, avec les générations de texte (avant/après) et les réponses aux questions.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 : télécharger et inspecter le corpus

On va regarder la longueur du texte, la ponctuation, etc.

In [2]:
from urllib.request import urlretrieve
import os

lien = ("https://raw.githubusercontent.com/karpathy/"
       "char-rnn/master/data/tinyshakespeare/input.txt")
urlretrieve(lien, "input.txt")
print("Fichier sauvegardé dans :", os.path.abspath("input.txt"))

with open("input.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

print("Taille du texte:", len(corpus))
print(corpus[:1000])

Fichier sauvegardé dans : /content/input.txt
Taille du texte: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the go

**Questions d'interprétation** — Quelle est ici l'unité d'un token ? Pourquoi préciser `encoding="utf-8"` ? Le fichier est-il déjà séparé en exemples ?

**Réponses :**
- Ici le token c'est juste un caractère (lettre, espace, saut de ligne etc). Au total on a 65 caractères différents.
- On précise utf-8 pour être sûr que python lise bien le fichier texte correctement, sinon selon l'ordi (par exemple sous Windows) ça peut bugger sur certains caractères spéciaux.
- Non le fichier n'est pas découpé, c'est juste un énorme bloc de texte de plus d'un million de caractères. C'est à nous de le découper pour créer les données d'entraînement.

## Étape 2 : passer des caractères aux entiers

In [3]:
caracteres_uniques = sorted(list(set(corpus)))
taille_vocab = len(caracteres_uniques)

char_to_id = {ch: i for i, ch in enumerate(caracteres_uniques)}
id_to_char = {i: ch for i, ch in enumerate(caracteres_uniques)}

texte_vers_ids = lambda s: [char_to_id[c] for c in s]
ids_vers_texte = lambda ids: ''.join([ id_to_char[i] for i in ids])

test_msg = "hii there"
print(texte_vers_ids(test_msg))
print(ids_vers_texte(texte_vers_ids(test_msg)))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


**Questions d'interprétation** — Pourquoi trier `chars` ? Que garantit le test `decode(encode(message)) == message` ?

**Réponses :**
- On trie la liste parce que la fonction set() n'est pas ordonnée de base. Si on ne trie pas, à chaque fois qu'on relance le notebook les lettres auront un id différent, et notre modèle ne sera plus compatible.
- Le test montre que notre encodage marche bien dans les deux sens et qu'on ne perd pas d'infos. En gros l'id correspond bien toujours au même caractère.

## Étape 3 : fabriquer les prédictions attendues

On divise le dataset: 90% pour l'entraînement et 10% pour la validation.

In [4]:
t_test = [1,2,3,4,5,6,7,8,9]
taille_bloc = 8
x_ex = t_test[:taille_bloc]
y_ex = t_test[1:taille_bloc + 1]
print("X:", x_ex)
print("Y:", y_ex)

X: [1, 2, 3, 4, 5, 6, 7, 8]
Y: [2, 3, 4, 5, 6, 7, 8, 9]


In [5]:
donnees_tensor = torch.tensor(texte_vers_ids(corpus), dtype=torch.long)

limite = int(0.9 * len(donnees_tensor))
set_entrainement = donnees_tensor[:limite]
set_validation = donnees_tensor[limite:]

taille_bloc = 8
seq_x = set_entrainement[:taille_bloc]
seq_y = set_entrainement[1:taille_bloc + 1]

for t in range(taille_bloc):
    contexte_actuel = seq_x[:t + 1]
    cible = seq_y[t]
    print(contexte_actuel, "->", cible)


tensor([18]) -> tensor(47)
tensor([18, 47]) -> tensor(56)
tensor([18, 47, 56]) -> tensor(57)
tensor([18, 47, 56, 57]) -> tensor(58)
tensor([18, 47, 56, 57, 58]) -> tensor(1)
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)


**Questions d'interprétation**
1. Combien de prédictions produit ce bloc ?
2. Pourquoi ne pas mélanger train et validation ?
3. Pour t = 3, quels sont le contexte et la cible ?

**Réponses :**
1. Ce bloc donne 8 prédictions au total (puisqu'on a choisi un block_size de 8).
2. Il ne faut pas mélanger train et validation sinon le modèle va juste apprendre le texte par coeur. On utilise la validation pour tester s'il arrive à généraliser sur des choses qu'il n'a jamais vues, ça permet de griller le surapprentissage.
3. A l'étape t=3, le contexte c'est [18, 47, 56, 57] et la cible à prédire c'est 58.

## Étape 4 : échantillonner un mini-batch

In [6]:
batch_size = 4
taille_bloc = 8

def recup_batch(split):
    donnees = set_entrainement if split == "train" else set_validation
    # On tire des indices de départ au hasard
    indices_debut = torch.randint(len(donnees) - taille_bloc, (batch_size,))

    xb = torch.stack([donnees[i : i + taille_bloc] for i in indices_debut])
    yb = torch.stack([donnees[i + 1 : i + taille_bloc + 1] for i in indices_debut])

    return xb, yb

x_batch, y_batch = recup_batch("train")
print(x_batch.shape, y_batch.shape)

torch.Size([4, 8]) torch.Size([4, 8])


**Questions d'interprétation**
1. Pourquoi retirer `block_size` à la borne de tirage ?
2. Que changerait B = 32 ?
3. Que changerait T = 64 ?

**Réponses :**
1. On retire la taille du bloc (block_size) pour être sûr de ne pas déborder à la toute fin du dataset quand on extrait la séquence et sa cible.
2. Si on passe à B=32, ça veut dire qu'on traite 32 exemples à la fois au lieu de 4. Ça consomme plus de mémoire mais la descente de gradient est plus stable car on fait une moyenne sur plus de textes en même temps.
3. Si on passe à T=64, le modèle peut regarder 64 caractères en arrière. Pour ce modèle bigramme simple ça ne change rien (il regarde que la dernière lettre de toute façon), mais pour les transformers des TP suivants ça augmente beaucoup la mémoire à court terme.

## Étape 5 : une table produit les logits

In [7]:
class ModeleBigramme(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Une table de taille (vocabulaire x vocabulaire)
        self.table_embeddings = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, cibles=None):
        logits = self.table_embeddings(idx)

        if cibles is None:
            loss = None
        else:
            b, t, c = logits.shape
            logits = logits.view(b * t, c)
            cibles = cibles.view(b * t)
            loss = F.cross_entropy(logits, cibles)

        return logits, loss

**Questions d'interprétation** — Pourquoi C × C ici ? Où retrouvera-t-on d_emb et la couche `lm_head` dans le TP 4 ?

**Réponses :**
- On a une table de C x C car les lignes correspondent au caractère actuel et les colonnes donnent les probabilités (logits) pour le caractère suivant. Comme il y a 65 caractères dans les deux cas, la matrice fait 65x65.
- Dans le TP4, cette opération est scindée en deux : d'abord on convertit le caractère en vecteur (dimension d_emb=64) avec une première table, puis on le repasse en probabilités sur les 65 caractères de sortie avec le lm_head (couche linéaire).

## Étapes 6 et 7 : générer, puis apprendre

### Étape 6 — génération (ajout de la méthode)

In [8]:
def fonction_generer(self, idx, max_nouveaux_tokens):
    for _ in range(max_nouveaux_tokens):
        logits, _ = self(idx)
        # on s'intéresse juste au dernier pas de temps
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        prochain_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, prochain_idx), dim=1)
    return idx

ModeleBigramme.generate = fonction_generer

In [9]:
# Génération avec des poids initiaux (complètement au hasard)
mon_modele = ModeleBigramme(taille_vocab)
start_idx = torch.zeros((1, 1), dtype=torch.long)
print("Avant entrainement:\n")
print(ids_vers_texte(mon_modele.generate(start_idx, max_nouveaux_tokens=100)[0].tolist()))

Avant entrainement:


;&BcvidMXG:DcJeW-',M'v mOfPnTMJeLRXYauWjYbIwmPO?
FYS.WNa&CwHxr-dYK-pn$Nd?Ny.CFbPMlOPcUxciVbvj.z
zRSq


### Étape 7 — entraînement

In [10]:
optimiseur = torch.optim.AdamW(mon_modele.parameters(), lr=1e-3)

for etape in range(1000):
    batch_x, batch_y = recup_batch("train")
    predictions, perte = mon_modele(batch_x, batch_y)

    optimiseur.zero_grad(set_to_none=True)
    perte.backward()
    optimiseur.step()

print(f"Loss à la fin : {perte.item():.4f}")

Loss à la fin : 3.8984


In [11]:
# Génération après l'entraînement
print("Après entrainement:\n")
print(ids_vers_texte(mon_modele.generate(start_idx, max_nouveaux_tokens=100)[0].tolist()))

Après entrainement:


f mj rDOEtHro rF3MwXgqsc!r.WNT;DNY'UxcMT!s$
IIg,RGfxBnTHWs!
;'X
KY3qybhX ,JoJaMzYtEVBB-vle.ed
BnxNyg


**Bilan du TP 1**
Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération ? Pourquoi aplatit-on B × T pendant la loss ?
Comparez la loss et la structure du texte avant/après.

**Bilan :**
- Lors de la génération, on garde juste le dernier pas de temps car c'est le seul qui correspond à la prédiction du nouveau caractère. Le reste on s'en fiche, c'est le texte qu'on connaît déjà.
- On aplatit le tenseur en B x T parce que la fonction cross_entropy de PyTorch a besoin d'une liste de prédictions "à plat" (une dimension pour les exemples, une pour les classes). On traite donc chaque prédiction indépendamment.
- Au niveau des résultats : au tout début, les poids sont tirés au hasard. La perte théorique pour un choix purement aléatoire parmi 65 caractères est de -ln(1/65) soit environ 4.17. Le modèle commence donc autour de cette valeur.
- Le texte généré avant entraînement ne ressemble à rien, c'est des caractères tapés au hasard.
- Après 1000 étapes, la loss descend un peu (vers 3.8 / 3.9). C'est pas énorme parce que le modèle est très simple et on a fait très peu d'étapes d'entraînement pour remplir toute la matrice 65x65. Le texte généré reste encore chaotique mais on commence tout doucement à voir des paires de lettres qui ressemblent un peu plus à des mots et des espaces.